# Interactive Dashboards

In the previous lessons, we built static charts and calculated the math behind them. But modern stakeholders rarely want a static PDF report; they want to *explore* the data themselves. They want to ask, *"What if I only look at the West region?"* or *"What if I drill down into December?"*

In this lesson, we learn how to transition from static reports to **Interactive Dashboards**. While you will often build these in tools like Tableau or PowerBI, we can simulate the exact same interactive logic right here in Python using **Plotly** and **Jupyter Widgets**.

An interactive dashboard shifts the power from the Data Scientist to the Business User. Instead of building 50 different charts for 50 different questions, you build one dynamic tool.

There are four main pillars of dashboard interactivity:
1. **Tooltips (Hover Details)**: Showing extra data only when the user asks for it.
2. **Filters & Slicers**: Allowing the user to include/exclude specific categories or date ranges.
3. **Drill-Downs (Hierarchies)**: Allowing the user to click a high-level summary to reveal granular details.
4. **Dashboard Actions (Cross-Filtering)**: Clicking a bar on one chart automatically filters every other chart on the screen.

Let's set up a Python sandbox with a complex retail dataset to see how we build these mechanics!

In [1]:
!pip install ipywidgets

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Create a rich dataset for interactivity
np.random.seed(42)

categories = ['Furniture', 'Technology', 'Office Supplies']
regions = ['North', 'South', 'East', 'West']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

df = pd.DataFrame({
    'Order_ID': range(1001, 1501),
    'Month': np.random.choice(months, 500),
    'Region': np.random.choice(regions, 500),
    'Category': np.random.choice(categories, 500, p=[0.2, 0.3, 0.5]),
    'Sales': np.random.randint(50, 2000, 500),
    'Profit_Margin': np.random.uniform(-0.2, 0.5, 500)
})

# Calculate absolute profit based on margin
df['Profit'] = df['Sales'] * df['Profit_Margin']

print("✅ Interactive Dataset Loaded!")
display(df.head(3))

✅ Interactive Dataset Loaded!


,Order_ID,Month,Region,Category,Sales,Profit_Margin,Profit
0,1001,Apr,North,Furniture,934,-0.006221,-5.809986
1,1002,May,South,Technology,1316,0.214645,282.473123
2,1003,Mar,South,Office Supplies,1566,0.438654,686.932699


# 1. Tooltips (Context on Demand)
If you put too much text on a chart, it becomes a cluttered mess (violating our Data-to-Ink ratio). 

**Tooltips** solve this. They allow you to hide highly specific details (like exact Order IDs or specific profit margins) behind a hover interaction. The screen stays clean, but the data is always just a mouse movement away.

In [3]:
# Create a Scatter Plot of Sales vs Profit
fig_hover = px.scatter(
    df, x='Sales', y='Profit', color='Category',
    title="Sales vs Profit (Hover for Details)",
    # THE TOOLTIP LOGIC:
    hover_name="Order_ID", # Makes the Order ID the bold title of the hover box
    hover_data={'Month': True, 'Region': True, 'Category': False} # Customize what shows up
)

# Add a zero-line to easily see what orders lost money
fig_hover.add_hline(y=0, line_dash="dot", annotation_text="Break Even", annotation_position="bottom right")

fig_hover.show()

*(Insight: If the CEO sees a massive negative outlier at the bottom of the chart, they don't have to call you to ask what order it was. They just hover their mouse over it and instantly see that Order #1245 in the West Region lost $300!)*

# 2. Filters & Slicers (User Control)
A **Filter** (often called a Slicer in PowerBI) is a dropdown menu, slider, or checklist that allows the user to subset the data. 

In Python, we can simulate this using `ipywidgets` to create a live dropdown menu that actively filters a Plotly chart!

In [4]:
# 1. Create a Dropdown Widget
region_dropdown = widgets.Dropdown(
    options=['All'] + list(df['Region'].unique()),
    value='All',
    description='Select Region:',
)

# 2. Create the function that updates the chart when the dropdown changes
def update_chart(selected_region):
    # Filter the data based on the dropdown
    if selected_region == 'All':
        filtered_df = df
    else:
        filtered_df = df[df['Region'] == selected_region]
        
    # Group the filtered data by Category
    category_sales = filtered_df.groupby('Category')['Sales'].sum().reset_index()
    
    # Draw the chart
    fig = px.bar(category_sales, x='Category', y='Sales', color='Category',
                 title=f"Total Sales for: {selected_region} Region")
    fig.show()

# 3. Link the Widget to the Function!
print("--- Interactive Filter Example ---")
widgets.interact(update_chart, selected_region=region_dropdown);

--- Interactive Filter Example ---


interactive(children=(Dropdown(description='Select Region:', options=('All', 'North', 'South', 'East', 'West')…

*(Insight: This single interactive element replaces the need to build 5 separate static charts (one for All, North, South, East, and West). It saves you development time and empowers the user.)*

# 3. Drill-Downs (Navigating Hierarchies)
Data naturally exists in hierarchies. For example: `Year -> Quarter -> Month` or `Country -> State -> City`. 

A **Drill-Down** allows a user to look at the high-level summary (e.g., Total Furniture Sales) and *click* on it to zoom into the next level of the hierarchy (e.g., Furniture Sales by Region). Plotly's **Sunburst** chart handles this natively!

In [5]:
# Group data to create a hierarchy: Category -> Region
hierarchy_df = df.groupby(['Category', 'Region'])['Sales'].sum().reset_index()

# Create an interactive Sunburst chart
fig_sunburst = px.sunburst(
    hierarchy_df, 
    path=['Category', 'Region'], # Define the Drill-Down path!
    values='Sales', 
    title="Click a Category to Drill Down into Regional Sales!"
)

fig_sunburst.show()

*(Insight: Try clicking on "Technology"! The chart smoothly animates, pushing the other categories out of the way to reveal exactly how Technology sales are distributed across the North, South, East, and West. Click the center circle to zoom back out.)*

# 4. Dashboard Actions (Cross-Filtering)
The holy grail of interactive dashboards in Tableau/PowerBI is **Cross-Filtering**. 

Imagine you have two charts on a screen: a Bar Chart of *Categories* and a Line Chart of *Sales over Time*. 
* If you click the "Furniture" bar on the first chart, the Line Chart automatically filters itself to *only* show the trend line for Furniture. 
* If you click "Technology," the Line Chart updates again. 

*(Note: While difficult to build in a static Jupyter Notebook, this is the default behavior when you combine multiple charts in an enterprise BI tool, or when you use Python web frameworks like **Streamlit** or **Plotly Dash**.)*

---

## Real-World Use Case or Analogy:
Think of an Interactive Dashboard like **Using Google Maps vs. a Paper Map**:

* **Static Reports (The Paper Map)**: You print out a map of the entire country. If you want to know what restaurants are on a specific street in Chicago, you have to buy a completely different, highly detailed map of Chicago. It is static, rigid, and cannot adapt to your changing questions.
* **Interactive Dashboard (Google Maps)**: 
    * **Filters**: You click a button that says "Only show me Gas Stations." (Slicer).
    * **Tooltips**: You hover your thumb over a red pin, and a tiny box pops up showing the name of the restaurant and its star rating.
    * **Drill-Down**: You pinch the screen to zoom in from the Country view (High-level) down to the State view, and finally down to the Street view (Granular). 

---